# 05 베이스라인 비교 및 메타데이터 효과 검증

04번에서 학습한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 실제로 좋은 모델인지 확인하기 위해 베이스라인과 비교한다.

목적은 다음과 같다.
비교를 위한 모델은 다음과 같습니다.
1. 'TF-IDF + Random Forest'
2. 'KcBERT PCA + MLP'
3. 'Metadata + MLP'

- 이후 실제 프로젝트 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)과 같은 지표로 성능을 비교한다.
- 메타데이터의 유무가 실제 성능에 기여하는지 확인한다.

이 노트북은 제안 모델의 단독 점수를 보고 끝내는 단계가 아니라, 계획서의 baseline 1/2/3과 04번 hybrid 모델을 같은 split과 같은 지표로 비교하는 검증 단계이다. 여기서 저장한 비교표와 모델 파일은 06번의 최종 모델 선택과 오답 분석 입력으로 사용된다.

In [1]:
import json
from itertools import combinations

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
MIN_TUNED_THRESHOLD = 0.5

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'emoji_count', 'photo_count']
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

text_train = raw_df.loc[train_idx, TEXT_COL].fillna('').astype(str)
text_val = raw_df.loc[val_idx, TEXT_COL].fillna('').astype(str)
text_test = raw_df.loc[test_idx, TEXT_COL].fillna('').astype(str)

print('Raw embedding data:', raw_df.shape)
print('Hybrid train:', X_train.shape)
print('Hybrid validation:', X_val.shape)
print('Hybrid test:', X_test.shape)
print('KcBERT feature 수:', len(emb_cols))
print('metadata feature:', meta_cols)
print('excluded feature:', ['rating'])
print('train label 분포:', y_train.value_counts().sort_index().to_dict())
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())

Raw embedding data: (8841, 782)
Hybrid train: (6188, 771)
Hybrid validation: (1326, 771)
Hybrid test: (1327, 771)
KcBERT feature 수: 768
metadata feature: ['text_length', 'emoji_count', 'photo_count']
excluded feature: ['rating']
train label 분포: {0: 3983, 1: 2205}
validation label 분포: {0: 854, 1: 472}
test label 분포: {0: 854, 1: 473}


/var/folders/wq/j2lqqp7n33dgrwj1r4q0nl9m0000gn/T/ipykernel_74865/435245180.py:6: DtypeWarning: Columns (0: store_name) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 공통 평가 함수 정의

- 기본 threshold는 `0.5`이다.
- validation 데이터에서 F1이 가장 높은 threshold를 추가로 찾으며, test 데이터는 threshold 선택에 사용하지 않는다.
- 단, threshold가 지나치게 낮아져 일반 리뷰를 과도하게 이벤트 리뷰로 오탐하지 않도록 `0.5 이상` 후보만 사용한다.

모든 모델에 같은 threshold 탐색 규칙을 적용하기 위해 평가 함수를 공통으로 둔다. validation에서 고른 threshold만 test에 적용하므로, test 성능을 보고 threshold를 고르는 누수를 피할 수 있다.


In [3]:
def tune_threshold_from_validation(y_true, prob, min_threshold=0.5):
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return float(min_threshold), pd.DataFrame()

    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    candidates = candidates[candidates['threshold'] >= min_threshold]

    if candidates.empty:
        return float(min_threshold), candidates

    best_threshold = float(candidates.sort_values('f1', ascending=False).iloc[0]['threshold'])
    return best_threshold, candidates.sort_values('f1', ascending=False).reset_index(drop=True)


def metric_row(model_name, feature_set, split, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': model_name,
        'feature_set': feature_set,
        'split': split,
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, pred)),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


def evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob):
    best_threshold, threshold_candidates = tune_threshold_from_validation(
        y_val,
        val_prob,
        min_threshold=MIN_TUNED_THRESHOLD,
    )

    rows = [
        metric_row(model_name, feature_set, 'validation_default_0_5', y_val, val_prob, 0.5),
        metric_row(model_name, feature_set, 'validation_tuned_min_0_5', y_val, val_prob, best_threshold),
        metric_row(model_name, feature_set, 'test_default_0_5', y_test, test_prob, 0.5),
        metric_row(model_name, feature_set, 'test_tuned_min_0_5', y_test, test_prob, best_threshold),
    ]
    return rows, best_threshold, threshold_candidates


## 4. 제안 모델 결과 로드

04번에서 이미 학습한 `KcBERT PCA + Metadata Hybrid + MLP` 결과를 불러온다.

이 결과가 05번 비교표에서 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 역할을 한다.

04번에서 이미 5-fold CV로 선택한 hybrid MLP를 다시 학습하지 않고, 저장된 성능표와 설정을 기준점으로 가져온다. 이후 baseline과 ablation 결과를 같은 표에 합쳐서 제안 모델이 실제로 비교 우위가 있는지 확인한다.


In [4]:
with open('outputs/proposed_mlp_selected_config.json', 'r', encoding='utf-8') as f:
    proposed_config = json.load(f)

proposed_bundle_for_metrics = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model_for_metrics = proposed_bundle_for_metrics['model']
proposed_feature_cols = proposed_bundle_for_metrics['feature_cols']
proposed_threshold = float(
    proposed_bundle_for_metrics.get(
        'best_threshold',
        proposed_config.get('best_threshold_from_validation', 0.5),
    )
)

proposed_val_prob = proposed_model_for_metrics.predict_proba(X_val[proposed_feature_cols])[:, 1]
proposed_test_prob = proposed_model_for_metrics.predict_proba(X_test[proposed_feature_cols])[:, 1]
proposed_metrics = pd.DataFrame([
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'validation_default_0_5', y_val, proposed_val_prob, 0.5),
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'validation_tuned_min_0_5', y_val, proposed_val_prob, proposed_threshold),
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'test_default_0_5', y_test, proposed_test_prob, 0.5),
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'test_tuned_min_0_5', y_test, proposed_test_prob, proposed_threshold),
])

proposed_metrics


,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_default_0_5,0.500000,0.533183,0.445837,0.385021,0.386047,0.527542,0.547512,458,396,223,249
1,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.500871,0.533937,0.446237,0.385021,0.386646,0.527542,0.547512,459,395,223,249
2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_default_0_5,0.500000,0.552374,0.440678,0.402899,0.397284,0.494715,0.554738,499,355,239,234
3,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.500871,0.553881,0.440454,0.402899,0.398291,0.492600,0.554738,502,352,240,233


## 5. TF-IDF + Random Forest 베이스라인

텍스트만 사용하는 전통적 베이스라인이다.

- 입력: `cleaned_review_text`
- feature: TF-IDF unigram/bigram
- classifier: `RandomForestClassifier`
- 불균형 보정: `class_weight='balanced'`

이 모델과 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 비교하면, KcBERT 임베딩과 메타데이터 결합 전략이 전통적 텍스트 베이스라인보다 나은지 확인할 수 있다.

PDF의 baseline 1에 해당하는 전통적 기법이다. 텍스트를 TF-IDF로 바꾼 뒤 Random Forest로 분류하므로, KcBERT 임베딩이나 메타데이터 없이도 텍스트 표면 패턴만으로 어느 정도 이벤트 리뷰를 잡을 수 있는지 확인하는 기준선 역할을 한다.


In [5]:
tfidf_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

tfidf_rf.fit(text_train, y_train)

tfidf_val_prob = tfidf_rf.predict_proba(text_val)[:, 1]
tfidf_test_prob = tfidf_rf.predict_proba(text_test)[:, 1]

tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates = evaluate_probabilities(
    'baseline_tfidf_random_forest',
    'cleaned_text_tfidf',
    y_val,
    tfidf_val_prob,
    y_test,
    tfidf_test_prob,
)

joblib.dump({
    'model': tfidf_rf,
    'model_name': 'baseline_tfidf_random_forest',
    'text_col': TEXT_COL,
    'target_col': LABEL_COL,
    'default_threshold': 0.5,
    'best_threshold': tfidf_best_threshold,
}, 'outputs/baseline_tfidf_random_forest_model.joblib')

print('TF-IDF + Random Forest best threshold:', tfidf_best_threshold)
display(pd.DataFrame(tfidf_rows).round(4))


TF-IDF + Random Forest best threshold: 0.5017506364189399


,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_default_0_5,0.5000,0.6312,0.2511,0.4006,0.4530,0.1737,0.5488,755,99,390,82
1,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_tuned_min_0_5,0.5018,0.6312,0.2511,0.4006,0.4530,0.1737,0.5488,755,99,390,82
2,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.6473,0.2866,0.4382,0.5137,0.1987,0.5852,765,89,379,94
3,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5018,0.6473,0.2866,0.4382,0.5137,0.1987,0.5852,765,89,379,94


## 6. KcBERT PCA only / Metadata only + MLP 비교

 제안 모델이 좋은 이유가 텍스트 임베딩 때문인지, 메타데이터 때문인지, 둘의 결합 때문인지 확인하기 위한 Ablation Study를 설계한다.

- `KcBERT PCA only + MLP`: KcBERT 임베딩 PCA feature만 사용
- `Metadata only + MLP`: metadata feature만 사용
- `KcBERT PCA + Metadata Hybrid + MLP`: 04번 결과 사용

MLP 구조와 optimizer, learning rate, alpha, activation, batch size, PCA 보존율, SMOTE k_neighbors는 04번에서 선택된 설정을 그대로 사용한다.

PDF의 baseline 2는 메타데이터만 입력한 MLP, baseline 3은 KcBERT 임베딩만 입력한 MLP에 해당한다. 두 ablation 모델을 04번에서 선택된 MLP 설정으로 맞추면, 성능 차이를 모델 설정 차이가 아니라 입력 feature 차이로 비교할 수 있다.


In [6]:
def make_mlp_model(cols, use_emb, metadata_cols=None, random_state=SEED):
    metadata_cols = list(metadata_cols or [])
    transformers = []
    if use_emb:
        transformers.append((
            'kcbert_pca',
            PCA(n_components=proposed_config.get('pca_n_components', 0.90), random_state=random_state),
            emb_cols,
        ))
    if metadata_cols:
        transformers.append(('metadata_scaler', StandardScaler(), metadata_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    return ImbPipeline([
        ('preprocess', preprocessor),
        ('smote', SMOTE(
            random_state=random_state,
            k_neighbors=proposed_config.get('smote_k_neighbors', 5),
        )),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=tuple(proposed_config['hidden_layer_sizes']),
            activation=proposed_config.get('activation', 'relu'),
            solver=proposed_config.get('solver', 'adam'),
            alpha=proposed_config['alpha'],
            batch_size=proposed_config.get('batch_size', 64),
            learning_rate_init=proposed_config.get('learning_rate_init', 1e-3),
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


def fit_mlp_variant(model_name, feature_set, cols, use_emb, metadata_cols=None):
    cols = list(cols)
    metadata_cols = list(metadata_cols or [])
    X_train_variant = X_train[cols]
    X_val_variant = X_val[cols]
    X_test_variant = X_test[cols]

    model = make_mlp_model(cols, use_emb=use_emb, metadata_cols=metadata_cols, random_state=SEED)
    model.fit(X_train_variant, y_train)

    val_prob = model.predict_proba(X_val_variant)[:, 1]
    test_prob = model.predict_proba(X_test_variant)[:, 1]

    rows, best_threshold, threshold_candidates = evaluate_probabilities(
        model_name,
        feature_set,
        y_val,
        val_prob,
        y_test,
        test_prob,
    )
    return model, rows, best_threshold, threshold_candidates


pca_only_model, pca_only_rows, pca_only_threshold, pca_only_candidates = fit_mlp_variant(
    'ablation_mlp_kcbert_pca_only',
    'kcbert_pca_only',
    emb_cols,
    use_emb=True,
    metadata_cols=[],
)

metadata_only_model, metadata_only_rows, metadata_only_threshold, metadata_only_candidates = fit_mlp_variant(
    'ablation_mlp_metadata_only',
    'metadata_only',
    meta_cols,
    use_emb=False,
    metadata_cols=meta_cols,
)

joblib.dump({
    'model': pca_only_model,
    'model_name': 'ablation_mlp_kcbert_pca_only',
    'feature_cols': emb_cols,
    'emb_cols': emb_cols,
    'meta_cols': [],
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'default_threshold': 0.5,
    'best_threshold': pca_only_threshold,
    'selected_config': proposed_config,
}, 'outputs/ablation_mlp_kcbert_pca_only_model.joblib')

joblib.dump({
    'model': metadata_only_model,
    'model_name': 'ablation_mlp_metadata_only',
    'feature_cols': meta_cols,
    'emb_cols': [],
    'meta_cols': meta_cols,
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'default_threshold': 0.5,
    'best_threshold': metadata_only_threshold,
    'selected_config': proposed_config,
}, 'outputs/ablation_mlp_metadata_only_model.joblib')

print('KcBERT PCA only best threshold:', pca_only_threshold)
print('Metadata only best threshold:', metadata_only_threshold)
display(pd.DataFrame(pca_only_rows + metadata_only_rows).round(4))


KcBERT PCA only best threshold: 0.5001755106686353
Metadata only best threshold: 0.5010779767831057


,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5002,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.5644,0.4516,0.4095,0.4096,0.5032,0.5633,511,343,235,238
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5002,0.5637,0.4501,0.4095,0.4086,0.5011,0.5633,511,343,236,237
4,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
5,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
6,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.4529,0.5027,0.3563,0.3718,0.7759,0.4957,234,620,106,367
7,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5011,0.4529,0.5014,0.3563,0.3713,0.7717,0.4957,236,618,108,365


## 7. 메타데이터 feature 조합 ablation

metadata-only 모델이 좋은 성능을 보였기 때문에, `text_length`, `emoji_count`, `photo_count`의 모든 비어 있지 않은 조합을 따로 학습해 어떤 행동 신호 조합이 성능에 기여했는지 확인한다.

이 실험은 baseline을 새로 정의하는 것이 아니라 metadata-only MLP 내부를 더 잘 해석하기 위한 ablation study이다. 각 조합은 04번에서 선택된 MLP 설정과 05번의 동일한 threshold 탐색 규칙을 사용한다.


In [7]:
metadata_ablation_specs = []
for size in range(1, len(meta_cols) + 1):
    for feature_tuple in combinations(meta_cols, size):
        feature_cols = list(feature_tuple)
        feature_slug = '_'.join(feature_cols)
        feature_label = '+'.join(feature_cols)
        metadata_ablation_specs.append({
            'model': f'metadata_ablation_mlp_{feature_slug}',
            'feature_set': f'metadata_ablation:{feature_label}',
            'feature_cols': feature_cols,
            'path': f'outputs/metadata_ablation_mlp_{feature_slug}_model.joblib',
            'input_type': 'tabular',
        })

metadata_ablation_rows = []
metadata_ablation_registry = []
metadata_ablation_thresholds = {}

for spec in metadata_ablation_specs:
    model, rows, threshold, threshold_candidates = fit_mlp_variant(
        spec['model'],
        spec['feature_set'],
        spec['feature_cols'],
        use_emb=False,
        metadata_cols=spec['feature_cols'],
    )
    metadata_ablation_rows.extend(rows)
    metadata_ablation_thresholds[spec['model']] = threshold
    metadata_ablation_registry.append({
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'path': spec['path'],
        'input_type': spec['input_type'],
    })

    joblib.dump({
        'model': model,
        'model_name': spec['model'],
        'feature_cols': spec['feature_cols'],
        'emb_cols': [],
        'meta_cols': spec['feature_cols'],
        'target_col': LABEL_COL,
        'input_type': 'tabular_raw',
        'default_threshold': 0.5,
        'best_threshold': threshold,
        'selected_config': proposed_config,
        'metadata_ablation': True,
    }, spec['path'])

metadata_ablation_metrics = pd.DataFrame(metadata_ablation_rows)
metadata_ablation_validation = (
    metadata_ablation_metrics[metadata_ablation_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
metadata_ablation_validation.insert(0, 'rank_f1', metadata_ablation_validation.index + 1)

metadata_ablation_accuracy_rank = (
    metadata_ablation_metrics[metadata_ablation_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['accuracy', 'f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
metadata_ablation_accuracy_rank.insert(0, 'rank_accuracy', metadata_ablation_accuracy_rank.index + 1)

best_worst_rows = []
project_metrics = ['f1', 'accuracy', 'pr_auc', 'precision', 'recall', 'roc_auc']
validation_metric_source = metadata_ablation_metrics[metadata_ablation_metrics['split'] == 'validation_tuned_min_0_5']
for metric in project_metrics:
    best = validation_metric_source.sort_values([metric, 'f1', 'pr_auc'], ascending=False).iloc[0]
    worst = validation_metric_source.sort_values([metric, 'f1', 'pr_auc'], ascending=True).iloc[0]
    best_worst_rows.append({
        'metric': metric,
        'type': 'best',
        'model': best['model'],
        'feature_set': best['feature_set'],
        'value': best[metric],
        'f1': best['f1'],
        'accuracy': best['accuracy'],
        'pr_auc': best['pr_auc'],
        'precision': best['precision'],
        'recall': best['recall'],
        'roc_auc': best['roc_auc'],
    })
    best_worst_rows.append({
        'metric': metric,
        'type': 'worst',
        'model': worst['model'],
        'feature_set': worst['feature_set'],
        'value': worst[metric],
        'f1': worst['f1'],
        'accuracy': worst['accuracy'],
        'pr_auc': worst['pr_auc'],
        'precision': worst['precision'],
        'recall': worst['recall'],
        'roc_auc': worst['roc_auc'],
    })

metadata_ablation_best_worst = pd.DataFrame(best_worst_rows)
metadata_ablation_metrics.to_csv('outputs/metadata_ablation_metrics.csv', index=False, encoding='utf-8-sig')
metadata_ablation_validation.to_csv('outputs/metadata_ablation_validation_f1_rank.csv', index=False, encoding='utf-8-sig')
metadata_ablation_accuracy_rank.to_csv('outputs/metadata_ablation_validation_accuracy_rank.csv', index=False, encoding='utf-8-sig')
metadata_ablation_best_worst.to_csv('outputs/metadata_ablation_best_worst.csv', index=False, encoding='utf-8-sig')

round_cols = {
    'threshold': 4,
    'accuracy': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
    'value': 4,
}

print(f'메타데이터 조합 수: {len(metadata_ablation_specs)}')
print('Validation F1 기준 메타데이터 조합 순위')
display(metadata_ablation_validation.round(round_cols))

print('Validation accuracy 기준 메타데이터 조합 순위')
display(metadata_ablation_accuracy_rank.round(round_cols))

print('프로젝트 평가 지표별 best/worst 메타데이터 조합')
display(metadata_ablation_best_worst.round(round_cols))

best_f1_row = metadata_ablation_validation.iloc[0]
worst_f1_row = metadata_ablation_validation.iloc[-1]
full_metadata_row = validation_metric_source[
    validation_metric_source['feature_set'] == 'metadata_ablation:text_length+emoji_count+photo_count'
]

print('메타데이터 조합 인사이트')
print(f"- F1 기준 최상 조합: {best_f1_row['feature_set']} (F1={best_f1_row['f1']:.4f}, accuracy={best_f1_row['accuracy']:.4f})")
print(f"- F1 기준 최하 조합: {worst_f1_row['feature_set']} (F1={worst_f1_row['f1']:.4f}, accuracy={worst_f1_row['accuracy']:.4f})")
if not full_metadata_row.empty:
    full_row = full_metadata_row.iloc[0]
    if best_f1_row['model'] != full_row['model']:
        print(
            '- 전체 메타데이터보다 일부 조합의 F1이 높다면, 제외된 feature가 현재 split에서는 noise처럼 작동했을 가능성이 있다.'
        )
    else:
        print('- 전체 메타데이터 조합이 가장 좋다면, 개별 행동 신호보다 조합 효과가 더 중요하다고 해석할 수 있다.')
print('- accuracy가 높아도 클래스 불균형 영향이 있으므로 최종 선택은 validation F1과 PR-AUC를 우선한다.')


메타데이터 조합 수: 7
Validation F1 기준 메타데이터 조합 순위


,rank_f1,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
1,2,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,validation_tuned_min_0_5,0.5077,0.4796,0.5222,0.3826,0.3879,0.7987,0.5558,259,595,95,377
2,3,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,validation_tuned_min_0_5,0.5031,0.4796,0.5215,0.3775,0.3876,0.7966,0.5476,260,594,96,376
3,4,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,validation_tuned_min_0_5,0.5002,0.3635,0.5188,0.3473,0.3549,0.9640,0.4795,27,827,17,455
4,5,metadata_ablation_mlp_text_length,metadata_ablation:text_length,validation_tuned_min_0_5,0.5000,0.4329,0.5085,0.3402,0.3677,0.8242,0.4932,185,669,83,389
5,6,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,validation_tuned_min_0_5,0.5004,0.5181,0.4697,0.3988,0.3861,0.5996,0.5548,404,450,189,283
6,7,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,validation_tuned_min_0_5,0.5002,0.5377,0.3344,0.3398,0.3430,0.3263,0.4943,559,295,318,154


Validation accuracy 기준 메타데이터 조합 순위


,rank_accuracy,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,validation_tuned_min_0_5,0.5002,0.5377,0.3344,0.3398,0.3430,0.3263,0.4943,559,295,318,154
1,2,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,validation_tuned_min_0_5,0.5004,0.5181,0.4697,0.3988,0.3861,0.5996,0.5548,404,450,189,283
2,3,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
3,4,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,validation_tuned_min_0_5,0.5077,0.4796,0.5222,0.3826,0.3879,0.7987,0.5558,259,595,95,377
4,5,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,validation_tuned_min_0_5,0.5031,0.4796,0.5215,0.3775,0.3876,0.7966,0.5476,260,594,96,376
5,6,metadata_ablation_mlp_text_length,metadata_ablation:text_length,validation_tuned_min_0_5,0.5000,0.4329,0.5085,0.3402,0.3677,0.8242,0.4932,185,669,83,389
6,7,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,validation_tuned_min_0_5,0.5002,0.3635,0.5188,0.3473,0.3549,0.9640,0.4795,27,827,17,455


프로젝트 평가 지표별 best/worst 메타데이터 조합


,metric,type,model,feature_set,value,f1,accuracy,pr_auc,precision,recall,roc_auc
0,f1,best,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,0.5227,0.5227,0.4849,0.3958,0.3900,0.7924,0.5579
1,f1,worst,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,0.3344,0.3344,0.5377,0.3398,0.3430,0.3263,0.4943
2,accuracy,best,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,0.5377,0.3344,0.5377,0.3398,0.3430,0.3263,0.4943
3,accuracy,worst,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,0.3635,0.5188,0.3635,0.3473,0.3549,0.9640,0.4795
4,pr_auc,best,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,0.3988,0.4697,0.5181,0.3988,0.3861,0.5996,0.5548
5,pr_auc,worst,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,0.3398,0.3344,0.5377,0.3398,0.3430,0.3263,0.4943
6,precision,best,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,0.3900,0.5227,0.4849,0.3958,0.3900,0.7924,0.5579
7,precision,worst,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,0.3430,0.3344,0.5377,0.3398,0.3430,0.3263,0.4943
8,recall,best,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,0.9640,0.5188,0.3635,0.3473,0.3549,0.9640,0.4795
9,recall,worst,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,0.3263,0.3344,0.5377,0.3398,0.3430,0.3263,0.4943


메타데이터 조합 인사이트
- F1 기준 최상 조합: metadata_ablation:text_length+emoji_count+photo_count (F1=0.5227, accuracy=0.4849)
- F1 기준 최하 조합: metadata_ablation:text_length+emoji_count (F1=0.3344, accuracy=0.5377)
- 전체 메타데이터 조합이 가장 좋다면, 개별 행동 신호보다 조합 효과가 더 중요하다고 해석할 수 있다.
- accuracy가 높아도 클래스 불균형 영향이 있으므로 최종 선택은 validation F1과 PR-AUC를 우선한다.


## 8. 메타데이터 영향도 분석

04번에서 저장한 제안 모델을 불러와 validation 데이터 기준 permutation importance를 계산한다.

메타데이터 컬럼을 하나씩 섞었을 때 F1이 얼마나 떨어지는지 확인한다. F1 감소폭이 클수록 해당 메타데이터가 모델 예측에 더 큰 영향을 준 것으로 해석한다.

이 분석은 모델 선택 기준이 아니라 사후 해석용이다. `text_length`, `emoji_count`, `photo_count` 중 어떤 행동 신호가 hybrid 모델 예측에 더 민감하게 작동했는지 확인한다.


In [8]:
def metadata_permutation_importance(model, X_val, y_val, metadata_cols, threshold, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_val, base_pred)

    rows = []
    for col in metadata_cols:
        drops = []
        for _ in range(repeats):
            X_perm = X_val.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            perm_f1 = f1_score(y_val, perm_pred)
            drops.append(base_f1 - perm_f1)

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })

    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


proposed_bundle = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model = proposed_bundle['model']
proposed_feature_cols = proposed_bundle['feature_cols']
proposed_threshold = proposed_bundle.get('best_threshold', proposed_config['best_threshold_from_validation'])

metadata_importance = metadata_permutation_importance(
    proposed_model,
    X_val[proposed_feature_cols],
    y_val,
    meta_cols,
    proposed_threshold,
    repeats=10,
)

metadata_importance_display = metadata_importance.copy()
metadata_importance_display.insert(0, 'rank', metadata_importance_display.index + 1)
metadata_importance_display = metadata_importance_display.rename(columns={
    'metadata': '메타데이터',
    'base_f1': '기준 F1',
    'mean_f1_drop': '평균 F1 감소폭',
    'std_f1_drop': 'F1 감소폭 표준편차',
    'repeats': '반복 횟수',
})
metadata_importance_display = metadata_importance_display.round({
    '기준 F1': 4,
    '평균 F1 감소폭': 4,
    'F1 감소폭 표준편차': 4,
})

top_metadata = metadata_importance.iloc[0]
print(
    f"가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
    f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
)
display(metadata_importance_display)

가장 큰 영향을 준 메타데이터: photo_count (평균 F1 감소폭: 0.0050)


,rank,메타데이터,기준 F1,평균 F1 감소폭,F1 감소폭 표준편차,반복 횟수
0,1,photo_count,0.4462,0.0050,0.0053,10
1,2,emoji_count,0.4462,-0.0007,0.0026,10
2,3,text_length,0.4462,-0.0012,0.0026,10


## 9. 전체 비교표 출력 및 모델 저장

04번 제안 모델 결과와 05번에서 학습한 baseline/ablation 결과를 하나의 표로 합친다.

모델 선택용 순위는 `validation_tuned_min_0_5` 기준을 우선 확인한다. `test_tuned_min_0_5`는 최종 선택 이후 일반화 성능을 확인하기 위한 참고 표로만 본다.

성능 비교 결과는 노트북 셀에 바로 출력하고, 모델 객체만 .joblib 파일로 따로 저장한다.

validation 순위는 06번의 최종 모델 선택에 사용하고, test 순위는 선택 이후 참고 성능으로만 본다. 이 분리를 유지해야 test 성능을 보고 모델을 고르는 평가 누수를 막을 수 있다.


In [9]:
baseline_metrics = pd.concat([
    proposed_metrics,
    pd.DataFrame(tfidf_rows),
    pd.DataFrame(pca_only_rows),
    pd.DataFrame(metadata_only_rows),
    metadata_ablation_metrics,
], ignore_index=True)

validation_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_summary.insert(0, 'rank', validation_summary.index + 1)

test_summary = (
    baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_summary.insert(0, 'rank', test_summary.index + 1)

validation_accuracy_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['accuracy', 'f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_accuracy_summary.insert(0, 'rank_accuracy', validation_accuracy_summary.index + 1)

round_cols = {
    'threshold': 4,
    'accuracy': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
}

validation_summary_display = validation_summary.round(round_cols)
test_summary_display = test_summary.round(round_cols)
validation_accuracy_summary_display = validation_accuracy_summary.round(round_cols)

baseline_metrics_display = (
    baseline_metrics
    .sort_values(['model', 'split'])
    .reset_index(drop=True)
    .round(round_cols)
)

baseline_metrics.to_csv('outputs/baseline_comparison_metrics.csv', index=False, encoding='utf-8-sig')
validation_summary.to_csv('outputs/baseline_validation_selection_summary.csv', index=False, encoding='utf-8-sig')
test_summary.to_csv('outputs/baseline_test_reference_summary.csv', index=False, encoding='utf-8-sig')
validation_accuracy_summary.to_csv('outputs/baseline_validation_accuracy_summary.csv', index=False, encoding='utf-8-sig')

saved_model_files = pd.DataFrame([
    {'model': 'baseline_tfidf_random_forest', 'feature_set': 'cleaned_text_tfidf', 'path': 'outputs/baseline_tfidf_random_forest_model.joblib', 'input_type': 'text'},
    {'model': 'ablation_mlp_kcbert_pca_only', 'feature_set': 'kcbert_pca_only', 'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib', 'input_type': 'tabular'},
    {'model': 'ablation_mlp_metadata_only', 'feature_set': 'metadata_only', 'path': 'outputs/ablation_mlp_metadata_only_model.joblib', 'input_type': 'tabular'},
    *metadata_ablation_registry,
    {'model': 'proposed_hybrid_mlp_04', 'feature_set': 'kcbert_pca+metadata', 'path': 'outputs/proposed_mlp_final_model.joblib', 'input_type': 'tabular'},
])
saved_model_files.to_csv('outputs/baseline_model_registry.csv', index=False, encoding='utf-8-sig')

print('Validation tuned 기준 모델 선택용 순위')
display(validation_summary_display)

print('Validation accuracy 기준 참고 순위')
display(validation_accuracy_summary_display)

print('Test tuned 기준 참고 순위')
display(test_summary_display)

print('Validation/Test 전체 평가 결과')
display(baseline_metrics_display)

print('저장된 모델 파일 registry')
display(saved_model_files)


Validation tuned 기준 모델 선택용 순위


,rank,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
1,2,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
2,3,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,validation_tuned_min_0_5,0.5077,0.4796,0.5222,0.3826,0.3879,0.7987,0.5558,259,595,95,377
3,4,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,validation_tuned_min_0_5,0.5031,0.4796,0.5215,0.3775,0.3876,0.7966,0.5476,260,594,96,376
4,5,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,validation_tuned_min_0_5,0.5002,0.3635,0.5188,0.3473,0.3549,0.9640,0.4795,27,827,17,455
5,6,metadata_ablation_mlp_text_length,metadata_ablation:text_length,validation_tuned_min_0_5,0.5000,0.4329,0.5085,0.3402,0.3677,0.8242,0.4932,185,669,83,389
6,7,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,validation_tuned_min_0_5,0.5004,0.5181,0.4697,0.3988,0.3861,0.5996,0.5548,404,450,189,283
7,8,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.5009,0.5339,0.4462,0.3850,0.3866,0.5275,0.5475,459,395,223,249
8,9,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5002,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
9,10,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,validation_tuned_min_0_5,0.5002,0.5377,0.3344,0.3398,0.3430,0.3263,0.4943,559,295,318,154


Validation accuracy 기준 참고 순위


,rank_accuracy,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_tuned_min_0_5,0.5018,0.6312,0.2511,0.4006,0.4530,0.1737,0.5488,755,99,390,82
1,2,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,validation_tuned_min_0_5,0.5002,0.5377,0.3344,0.3398,0.3430,0.3263,0.4943,559,295,318,154
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5002,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
3,4,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.5009,0.5339,0.4462,0.3850,0.3866,0.5275,0.5475,459,395,223,249
4,5,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,validation_tuned_min_0_5,0.5004,0.5181,0.4697,0.3988,0.3861,0.5996,0.5548,404,450,189,283
5,6,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
6,7,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
7,8,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,validation_tuned_min_0_5,0.5077,0.4796,0.5222,0.3826,0.3879,0.7987,0.5558,259,595,95,377
8,9,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,validation_tuned_min_0_5,0.5031,0.4796,0.5215,0.3775,0.3876,0.7966,0.5476,260,594,96,376
9,10,metadata_ablation_mlp_text_length,metadata_ablation:text_length,validation_tuned_min_0_5,0.5000,0.4329,0.5085,0.3402,0.3677,0.8242,0.4932,185,669,83,389


Test tuned 기준 참고 순위


,rank,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,test_tuned_min_0_5,0.5002,0.3580,0.5186,0.3545,0.3539,0.9704,0.4959,16,838,14,459
1,2,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,test_tuned_min_0_5,0.5031,0.4521,0.5038,0.3687,0.3720,0.7801,0.5248,231,623,104,369
2,3,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,test_tuned_min_0_5,0.5077,0.4514,0.5021,0.3688,0.3711,0.7759,0.5250,232,622,106,367
3,4,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5011,0.4529,0.5014,0.3563,0.3713,0.7717,0.4957,236,618,108,365
4,5,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,test_tuned_min_0_5,0.5011,0.4529,0.5014,0.3563,0.3713,0.7717,0.4957,236,618,108,365
5,6,metadata_ablation_mlp_text_length,metadata_ablation:text_length,test_tuned_min_0_5,0.5000,0.4137,0.5000,0.3879,0.3592,0.8224,0.5401,160,694,84,389
6,7,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5002,0.5637,0.4501,0.4095,0.4086,0.5011,0.5633,511,343,236,237
7,8,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.5009,0.5539,0.4405,0.4029,0.3983,0.4926,0.5547,502,352,240,233
8,9,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,test_tuned_min_0_5,0.5004,0.4778,0.4052,0.3656,0.3410,0.4989,0.5010,398,456,237,236
9,10,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,test_tuned_min_0_5,0.5002,0.5328,0.3049,0.3355,0.3246,0.2875,0.4506,571,283,337,136


Validation/Test 전체 평가 결과


,model,feature_set,split,threshold,accuracy,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.5644,0.4516,0.4095,0.4096,0.5032,0.5633,511,343,235,238
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5002,0.5637,0.4501,0.4095,0.4086,0.5011,0.5633,511,343,236,237
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5002,0.5354,0.4420,0.3679,0.3861,0.5169,0.5134,466,388,228,244
4,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.4529,0.5027,0.3563,0.3718,0.7759,0.4957,234,620,106,367
5,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5011,0.4529,0.5014,0.3563,0.3713,0.7717,0.4957,236,618,108,365
6,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
7,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5011,0.4849,0.5227,0.3958,0.3900,0.7924,0.5579,269,585,98,374
8,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.6473,0.2866,0.4382,0.5137,0.1987,0.5852,765,89,379,94
9,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5018,0.6473,0.2866,0.4382,0.5137,0.1987,0.5852,765,89,379,94


저장된 모델 파일 registry


,model,feature_set,path,input_type
0,baseline_tfidf_random_forest,cleaned_text_tfidf,outputs/baseline_tfidf_random_forest_model.joblib,text
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,outputs/ablation_mlp_kcbert_pca_only_model.joblib,tabular
2,ablation_mlp_metadata_only,metadata_only,outputs/ablation_mlp_metadata_only_model.joblib,tabular
3,metadata_ablation_mlp_text_length,metadata_ablation:text_length,outputs/metadata_ablation_mlp_text_length_mode...,tabular
4,metadata_ablation_mlp_emoji_count,metadata_ablation:emoji_count,outputs/metadata_ablation_mlp_emoji_count_mode...,tabular
5,metadata_ablation_mlp_photo_count,metadata_ablation:photo_count,outputs/metadata_ablation_mlp_photo_count_mode...,tabular
6,metadata_ablation_mlp_text_length_emoji_count,metadata_ablation:text_length+emoji_count,outputs/metadata_ablation_mlp_text_length_emoj...,tabular
7,metadata_ablation_mlp_text_length_photo_count,metadata_ablation:text_length+photo_count,outputs/metadata_ablation_mlp_text_length_phot...,tabular
8,metadata_ablation_mlp_emoji_count_photo_count,metadata_ablation:emoji_count+photo_count,outputs/metadata_ablation_mlp_emoji_count_phot...,tabular
9,metadata_ablation_mlp_text_length_emoji_count_...,metadata_ablation:text_length+emoji_count+phot...,outputs/metadata_ablation_mlp_text_length_emoj...,tabular


## 10. 05번 결과 해석 기준

05번 실행 후에는 다음을 확인한다.

1. validation 기준에서 제안 모델이 `baseline_tfidf_random_forest`보다 좋은가?
2. validation 기준에서 제안 모델이 `ablation_mlp_kcbert_pca_only`보다 좋은가?
3. validation 기준에서 제안 모델이 `ablation_mlp_metadata_only`보다 좋은가?
4. metadata를 결합했을 때 F1, PR-AUC, Recall이 실제로 올라갔는가?
5. 메타데이터 조합 ablation에서 어떤 조합이 F1/accuracy 기준으로 가장 좋고 나쁜가?

이 결과가 06번의 입력이 된다.

06번에서는 다음 작업을 진행한다.

- validation 기준 최종 모델 선택
- 선택 모델의 test 성능 최종 확인
- FP/FN 오답 리뷰 추출
- 은어, 신조어, 우회 표현에 대한 실패 사례 분석
- 메타데이터가 텍스트 기반 모호성을 줄였는지 해석

따라서 05번은 별점 정제 단계가 아니라, 제안 모델의 타당성을 검증하는 단계이다.

이 해석 기준을 먼저 고정해 두면 06번에서 validation F1 중심으로 최종 모델을 선택하고, test 데이터는 오답 분석과 참고 성능 확인에만 사용할 수 있다.
